In [1]:
!pip install -r requirement.txt

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
from model_mel_input.spectttra import SpecTTTra
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from model_mel_input.dataset import SonicDataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


/home/admin/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
hyper_param_5s = [
    { # alpha
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 1,
        "t_clip": 3,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # beta
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 3,
        "t_clip": 5,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # gamma
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 5,
        "t_clip": 7,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    }
]

hyper_param_120s = [
    { # alpha
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 1,
        "t_clip": 3,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # beta
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 3,
        "t_clip": 5,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # gamma
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 5,
        "t_clip": 7,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    }
]

In [5]:
df = pd.read_csv(f"crop_data/crop{5}.csv")
data = SonicDataset(df)

train_loader = DataLoader(data, batch_size=32, shuffle=True, num_workers=2)
batch = next(iter(train_loader))
X = batch["x"]
y = batch["y"]
print(X.shape, y.shape)


torch.Size([32, 1, 128, 157]) torch.Size([32])


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10537 entries, 0 to 10536
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   filename   10537 non-null  object
 1   path       10537 non-null  object
 2   fake_type  10537 non-null  object
 3   label      10537 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 329.4+ KB


In [7]:
train_loss_hist, train_acc_hist, test_acc_hist = [], [], []
num_epoch = 5

In [10]:
def train(len_clip, version = 1):
    def get_model_param(len_clip, version):
        if len_clip == 5:
            return hyper_param_5s[version-1]
        return hyper_param_120s[version-1]

    if len_clip not in [5, 120]:
        raise ValueError("Clip len only 5s or 120s")
    if version not in [1, 2, 3]:
        raise ValueError("Not availible model version")

    df = pd.read_csv(f"crop_data/crop{len_clip}.csv")

    df_train, df_test = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    print(f"Train data: {df_train.shape}")
    print(f"Test_data: {df_test.shape}")

    train_data = SonicDataset(df_train)
    test_data = SonicDataset(df_test)

    # sample = train_data[0]
    # print(sample.keys())
    # print(type(sample["x"]), sample["x"].shape)
    # print(type(sample["y"]), sample["y"])
    # print(type(sample["filename"]), sample["filename"])
    # print(type(train_data))

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

    # Infer real temporal dimension T from data to avoid Conv1d channel mismatch.
    sample_batch = next(iter(train_loader))
    inferred_temp_dim = sample_batch["x"].shape[-1]

    hyper_param = get_model_param(len_clip, version).copy()
    hyper_param["input_temp_dim"] = inferred_temp_dim
    print(f"Using input_temp_dim={inferred_temp_dim}")

    model = SpecTTTra(**hyper_param).to(device)

    loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters())
    cnt = 0
    for epoch in range(num_epoch):
        # train
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_sample = 0
        
        for batch in train_loader:
            X = batch["x"].to(device)
            y = batch["y"].to(device).float().unsqueeze(1)

            optimizer.zero_grad()

            logits = model(X)
            cur_loss = loss(logits, y)
            cur_loss.backward()
            optimizer.step()

            total_loss += cur_loss.item() * X.size(0)
            total_sample += X.size(0)

            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (pred == y).sum().item()

            print(cnt)
            cnt = cnt + 1

        epoch_loss = total_loss / total_sample
        epoch_acc = total_correct / total_sample

        # eval
        model.eval()
        test_sample = 0
        test_correct = 0
        with torch.no_grad():
            for batch in test_loader:
                X = batch["x"].to(device)
                y = batch["y"].to(device).float().unsqueeze(1)

                logits = model(X)
                pred = (torch.sigmoid(logits) >= 0.5).float()
                test_sample += X.size(0)
                test_correct += (y == pred).sum().item()

        test_acc = test_correct / test_sample

        train_loss_hist.append(epoch_loss)
        train_acc_hist.append(epoch_acc)
        test_acc_hist.append(test_acc)

        print(f"Epoch [{epoch+1}/{num_epoch}] "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Train Acc: {epoch_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}")

        return model

In [11]:
model = train(5)

Train data: (8429, 4)
Test_data: (2108, 4)
Using input_temp_dim=157
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
26